# Imports, config, session and utilities

In [2]:
from pathlib import Path
import sys

from snowflake.core import Root
from snowflake.snowpark.session import Session

In [3]:
current_file_path = Path().resolve()
sys.path.insert(0, str(current_file_path.parent))

In [4]:
# Custom python file import
# Import statement after adding parent file path to recognize src python files
from src.support_agent.config import get_settings
from src.support_agent.snowflake_client import create_snowpark_session

In [5]:
# Getting .env config and init snowflake session
settings = get_settings()
session = create_snowpark_session(settings)

# Build Search Service

Create and test Cortex Search service.

## Setup search service


- Setup curated tickets table 
- Create search service pointing on curated table

In [52]:
create_curated_table_query = """
CREATE TABLE IF NOT EXISTS TICKETS (
--    ticket_id VARCHAR PRIMARY KEY,
    language VARCHAR,
    subject VARCHAR,
    body VARCHAR,
    answer VARCHAR,
--    created_at TIMESTAMP,
--    category VARCHAR,
    priority VARCHAR,
    queue VARCHAR,
    type VARCHAR
--  status VARCHAR,
--  tags ARRAY
--  product VARCHAR,
--  topic VARCHAR
);
"""

In [53]:
query_res = session.sql(create_curated_table_query).collect()
query_res

[Row(status='Table TICKETS successfully created.')]

In [54]:
insert_tickets_query = """
INSERT INTO TICKETS
SELECT
--    'NULL' AS ticket_id,
    language,
    subject,
    body,
    answer,
--    created_at,
--    category,
    priority,
    queue,
    type
--  status,
--  'NULL' as tags, -- SPLIT(tags, ',')
--  NULL AS product,
--  NULL AS topic

FROM RAW.SUPPORT_TICKETS;
"""

In [55]:
query_res = session.sql(insert_tickets_query).collect()
query_res

[Row(number of rows inserted=28587)]

In [56]:
create_search_service_query = """
CREATE OR REPLACE CORTEX SEARCH SERVICE support_tickets_search_service
  ON body_answer
  ATTRIBUTES subject, priority, type, queue, language
  WAREHOUSE = compute_wh
  TARGET_LAG = '1 day'
  EMBEDDING_MODEL = 'snowflake-arctic-embed-l-v2.0'
  AS (
    SELECT *,
       concat('Body: ', body, ' \n Answer:', answer) as body_answer
    FROM CURATED.TICKETS
);
"""

In [57]:
query_res = session.sql(create_search_service_query).collect()
query_res

[Row(status='Cortex search service SUPPORT_TICKETS_SEARCH_SERVICE successfully created.')]

## Test search service with custom queries

In [ ]:
class CortexSearchRetriever:
    """Search service using snowpark session."""

    def __init__(self, snowpark_session: Session, limit_to_retrieve: int = 4):
        self._snowpark_session = snowpark_session
        self._limit_to_retrieve = limit_to_retrieve

    def retrieve(self, query: str) -> list[str]:
        """Retrieve method call to run search service on query."""
        root = Root(session)

        search_service = (
            root.databases["TAL_DB"]
            .schemas["AGENT"]
            .cortex_search_services["support_tickets_search_service"]
        )
        resp = search_service.search(
            query=query, columns=["body"], limit=self._limit_to_retrieve
        )

        if resp.results:
            return [curr["body"] for curr in resp.results]
        return []


retriever = CortexSearchRetriever(snowpark_session=session, limit_to_retrieve=3)
retrieved_context = retriever.retrieve(query="database optimization")
retrieved_context

['require optimization of database queries for the project management platform to enhance performance and scalability, and improve the user experience to support growth.',
 'require optimization of database queries for the project management platform to enhance performance and scalability. also, aim to improve the user experience to support growth. please provide more details on the current database setup and any specific issues encountered for better understanding of your needs.',
 'require optimization of database queries for the project management platform to boost performance and scalability. aim to enhance user experience and support growth.']

# Close session 

In [6]:
session.close()